# LangChain & Ollama Introduction

A practical introduction to core LangChain concepts and workflows:
- **Agent & Tool Integration**: Building and invoking agents with `create_agent` and custom `@tool` definitions.
- **Model Providers**: Working with local LLMs via `ChatOllama` and cloud providers like Google GenAI.
- **Streaming & Batching**: Real-time token streaming and concurrent batch inference.
- **Tool Binding**: Manual tool binding with `bind_tools` and execution loops.
- **Structured Message Handling**: Multi-turn conversations using `SystemMessage`, `HumanMessage`, and `AIMessage`.

In [ ]:
import langchain_ollama
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
load_dotenv()
from langchain.agents import create_agent
import os

os.environ["GEMINI_API_KEY"]=str(os.getenv("GEMINI_API_KEY"))
def get_weather(city:str)->str:
    """get weather from city"""
    return f"weather in {city} is sunny"

# agent = create_agent(
#     model="google_genai:gemini-3.8-flash",
#     tools=[get_weather],
#     system_prompt="You are a helpful assistant. Use tools when needed.",
# )
llm = ChatOllama(model="qwen3:4b", temperature=0, reasoning=False)

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant. Use tools when needed.",
)

agent

### 1. Agent Initialization & Tool Definition

- **Tool Creation**: Defines a basic `get_weather` tool with type hints and docstring.
- **Model Configuration**: Connects to the local Ollama instance running `qwen3:4b` via `ChatOllama`.
- **Agent Compilation**: Uses `create_agent` to build a LangGraph-compiled state graph capable of orchestrating model decisions and calling tools.

In [ ]:
result = agent.invoke({"messages":[{"content": "what is weather in Pune", "role": "user"}]})
result["messages"]


### 2. Agent Invocation & Message History

- **Input**: Passes a user message querying the weather in Pune.
- **Execution Flow**: The model detects that a tool is needed, generates a tool call, executes `get_weather`, receives a `ToolMessage`, and synthesizes the final response.
- **Output Trace**: Inspects `result["messages"]` showing the complete history of intermediate agent steps and outputs.

In [ ]:
# from langchain.chat_models import init_chat_model
# chat_model = init_chat_model("google_genai:gemini-3.8-flash")
# resp = chat_model.invoke("why do dogs bark?")

### 3. Alternate Model Initialization (Generic Interface)

- Demonstrates using LangChain's universal `init_chat_model` interface to switch between model providers dynamically without changing application code.

In [ ]:
#resp.content

### 4. Inspecting Model Content

- Shows accessing the raw text output using `.content` on the response object returned by the model invocation.

In [ ]:
# from langchain_google_genai import ChatGoogleGenerativeAI
# model = ChatGoogleGenerativeAI(model="gemini-3.8-flash")
# resp = model.invoke("why do dogs bark?")
# resp.content

### 5. Provider-Specific Model Initialization (Google GenAI)

- Illustrates direct integration with Google Generative AI via `ChatGoogleGenerativeAI` (`gemini-3.8-flash`).

In [ ]:
model = ChatOllama(model="qwen3:4b", temperature=0, reasoning=False)

### 6. Local ChatOllama Setup

- Instantiates `ChatOllama` using the local `qwen3:4b` model with `temperature=0` for deterministic outputs and `reasoning=False`.

In [ ]:
for chunk in model.stream("on how many devices we can implement offline llm with chatbot?"):
    print(chunk.text, end="|", flush=True)

### 7. Streaming Response Tokens

- Demonstrates `.stream()` on a single prompt, printing response token chunks in real time as they arrive from Ollama.

In [ ]:
response = model.batch(
    ["on how many devices we can implement offline llm",
    "what are limitations?",
    "can we implement custom chatbots everywhere?"],
)

for resp in response:
    print(resp.text, end="|", flush=True)

### 8. Batch Processing

- Uses `model.batch()` to submit a list of independent queries simultaneously, returning a list of responses.

In [ ]:
from langchain_core.runnables import RunnableConfig

model.batch(
    [
        "on how many devices we can implement offline llm",
        "what are limitations?",
        "can we implement custom chatbots everywhere?"
    ],
    config=RunnableConfig(max_concurrency=5),
)

### 9. Batch Processing with Concurrency Control

- Utilizes `config={"max_concurrency": 5}` to limit and optimize parallel request execution during batch operations.

In [ ]:
from langchain.tools import tool

model = ChatOllama(model="qwen3:4b", temperature=0, reasoning=False)
@tool
def get_weather(city:str)->str:
    """get weather from city"""
    return f"it's sunny in {city}"

messages = [{"role":"user", "content":"what is weather in Pune"}]
model_with_tools = model.bind_tools([get_weather])
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tools in ai_msg.tool_calls:
    tool_message = get_weather.invoke(tools)
    messages.append(tool_message)

final_resp = model_with_tools.invoke(messages)


### 10. Manual Tool Binding and Execution Loop

- Decorates `get_weather` as a tool and binds it to `ChatOllama` with reasoning enabled using `model.bind_tools([get_weather])`.
- Shows manual loop execution: invoking the model, inspecting `ai_msg.tool_calls`, executing the tool, appending `tool_message` to history, and re-invoking the model.

In [ ]:
final_resp.text

### 11. Final Response Inspection

- Displays `final_resp.text` from the manual tool execution flow, containing the LLM's final synthesized response based on tool results.

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage
messages=[
    SystemMessage("""
    You are AI expert with experience in UI/UX development and rest api with java and python.
    you are also expert in linux system.
    you provide solutions with proper digrams and technologies(nodejs, rest, python, java
    """),
    HumanMessage("What a the best way to use custom UI for LLM like ollama?"),
    AIMessage("hi, I am AI assistant, how may I help you?")
]
resp = model.invoke(messages)
resp.text

### 12. Structured Multi-Turn Conversation History

- Constructs a structured message history using `SystemMessage`, `AIMessage`, and `HumanMessage`.
- Preserves context across dialogue turns and invokes the model with the multi-message sequence.

In [ ]:
for chunk in model.stream(messages):
    print(chunk.text, end="|", flush=True)

### 13. Streaming Multi-Turn Conversations

- Streams tokens in real time from a multi-turn conversation sequence using `model.stream(messages)`.